# ISOM 835 · Midterm Model Competition — Starter Notebook
**Hotel cancellations · launches Mon Oct 19 · leaderboard closes Mon Nov 9, 4:30 PM · Prof. Hasan Arslan**

This notebook gets a leak-free submission on the board in ten minutes. It reproduces the **logistic baseline** benchmark exactly; everything after that is yours.

> **Frame it.** *Unit:* one booking · *Target:* `is_canceled` · *Metric:* ROC-AUC on a hidden half of `test.csv` · *Decision (for the memo):* overbooking and deposit policy.

**Setup.** *On Kaggle (fastest):* in the competition's **Code** tab click **New Notebook** (the data attaches itself), then **File → Import Notebook → Link**, paste `https://raw.githubusercontent.com/harslan/isom-835/master/public/session-05/ISOM835_Competition_Starter.ipynb`, and **Run All**. *In Colab:* download `train.csv`, `test.csv`, and `sample_submission.csv` from the competition's Data tab and upload them here (folder icon on the left → upload). The next cells find the files either way.

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

from pathlib import Path
def find(name):  # Colab: the file you uploaded · Kaggle: the competition data under /kaggle/input
    for p in [Path(name), *sorted(Path('/kaggle/input').glob(f'**/{name}'))]:
        if p.exists(): return p
    raise FileNotFoundError(f'{name} not found: upload it (Colab) or add the competition data (Kaggle)')

train = pd.read_csv(find('train.csv')); test = pd.read_csv(find('test.csv'))
print(train.shape, test.shape, f'cancel rate {train.is_canceled.mean():.3f}')
X, y = train.drop(columns=['id', 'is_canceled']), train['is_canceled']

## 1. Look before you model

Before you build anything, get to know the columns. A good model on bad columns still gives bad answers. Look for three kinds of trouble.

**Trouble 1: columns that give away the answer ("leaks"). We already removed these.**
The original data had `reservation_status` ("Canceled" or "Check-Out"), the date of that status, and `assigned_room_type` (the room the guest actually got at check-in). All three are known only *after* you know whether the booking was canceled. Using them would be like predicting a game with tomorrow's newspaper: a great score on paper, and useless to a real hotel. They are gone from `train.csv` and `test.csv`.

**Trouble 2: ID numbers that look like amounts: `agent` and `company`.**
`agent` is the travel agent's ID number, and `company` is the ID of the company that made the booking. These are **labels, not quantities**: agent 240 is not "more" than agent 9, just a different agent. But they are stored as numbers, so this starter treats them as numbers, as if a bigger ID meant something. They are also often blank (`company` in 94% of bookings, `agent` in 14%), and a blank usually means "no agent, booked directly," which is useful information in itself.
→ This starter leaves them as they are so that it reproduces the Kaggle benchmark exactly. **Fixing them is your first easy win** (see Section 4).

**Trouble 3: a text column with too many values: `country`.**
`country` has 167 different values. Five countries make up 73% of all bookings, and 99 countries have fewer than 20 bookings each. One-hot encoding (Session 3) makes one 0/1 column per country, and a column built from 3 bookings teaches the model noise, not a pattern. That's why the pipeline below uses `min_frequency=20`: every country with fewer than 20 bookings goes into one shared "other" column.

**The next cell runs three quick checks:** how much of each column is missing, the cancel rate by deposit type, and the cancel rate by how far ahead the booking was made.

In [ ]:
print('Share of each column that is missing (top 6):')
print(X.isna().mean().sort_values(ascending=False).head(6).round(3), '\n')

print('Cancel rate by deposit type:')
print(train.groupby('deposit_type')['is_canceled'].mean().round(3), '\n')

print('Cancel rate by lead time (days between booking and arrival):')
print(train.groupby(pd.cut(train['lead_time'], [-1, 7, 30, 90, 180, 800]), observed=True)['is_canceled'].mean().round(3))

**What you should see, and what it means:**
- **Missing values:** `company` is blank 94% of the time and `agent` 14%; almost everything else is complete. The pipeline fills blanks for you (the median for numbers, the word "missing" for text).
- **Deposit type:** bookings with a **Non Refund** deposit cancel **99%** of the time. That's backwards: why would someone who paid a non-refundable deposit cancel? Don't just use it; ask why. It's one of the puzzles worth a paragraph in your memo.
- **Lead time:** bookings made within a week cancel about **10%** of the time, and bookings made more than six months ahead about **57%**. The further ahead people book, the more likely plans change. It's a strong, sensible signal, and exactly the kind of feature you want.

## 2. The Session 3 pipeline, unchanged

In [ ]:
num = X.select_dtypes(include='number').columns.tolist()
cat = [c for c in X.columns if c not in num]
prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')), ('oh', OneHotEncoder(handle_unknown='ignore', min_frequency=20))]), cat),
])
model = Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=3000))])
cv = StratifiedKFold(5, shuffle=True, random_state=835)
scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'5-fold AUC: {scores.mean():.4f} ± {scores.std():.4f}   ← trust this more than the public leaderboard')

## 3. Fit on everything, write the submission

In [ ]:
model.fit(X, y)
sub = pd.DataFrame({'id': test['id'], 'is_canceled': model.predict_proba(test.drop(columns=['id']))[:, 1]})
sub.to_csv('submission.csv', index=False)
print(sub.head(), sub.shape)
# Kaggle: Save Version (Save & Run All), open the saved notebook's Output tab, click Submit to Competition.
# Colab: download submission.csv (folder icon → ⋮ → Download) and upload it with Submit Prediction. Expect ≈ 0.89 AUC.

## 4. Where the points are
1. **Treat `agent` and `company` as labels, not numbers** (Trouble 2 above). Before building the pipeline, turn them into text, with 0 meaning "none": `X[col] = X[col].fillna(0).astype(int).astype(str)` for each of the two columns, and do the same to `test`. They then go through one-hot encoding like `country`. On our check, this alone lifts cross-validated AUC from about 0.889 to about 0.907, with the same model.
2. **Swap the last step** for `HistGradientBoostingClassifier` (Session 8) with early stopping — the reference lands near 0.96.
3. **Engineer features** (Session 3): total nights, weekend share, `previous_cancellations > 0`, lead-time buckets, room-type mismatch is gone but `booking_changes` and `total_of_special_requests` are not.
4. **Validate like production** (Session 9): if you suspect repeated agents, try `GroupKFold` by `agent` and see whether your CV score holds.
5. **Do not tune on the public leaderboard.** Five submissions a day is plenty if your CV is honest.

Your memo (one page, due Mon Nov 9, 11:59 PM): what you tried, what worked, what leaked, and one thing you would do with two more weeks.